# Email Open and Click Behavior Analysis

Analyze email campaign data to identify what affects open rate, click rate, and
disengagement. Covers send-time patterns, subject-line effectiveness, recipient
journey segmentation, list-churn proxies, and conversion funnels.


## What This Notebook Answers

- Which campaigns get the best open rates, click rates, and click-to-open ratios?
- Does the hour or day of send affect engagement?
- Which subject-line tone (welcome, personalization, urgency/discount, transactional)
  drives more opens vs more clicks?
- Which audience segments are highly engaged vs dormant?
- Which recipients show "unsubscribe-like" behavior (churn from the mailing list)?
- How long after sending do opens and clicks typically arrive?


## Dataset

**Source:** Wanderlust Adventures — anonymized email engagement log from Kaggle
(`mariusnikiforovas/email-marketing-campaign-dashboard`).

Each row is one email delivery to one recipient. The key columns are:

| Column | Description |
|---|---|
| `email_name` | Campaign name / subject proxy (4 distinct campaigns) |
| `sent_date` | Datetime the email was sent |
| `open_date` | Datetime the recipient opened the email (NaT = not opened) |
| `click_date` | Datetime of first click (NaT = no click) |
| `bounce_date` | Datetime of bounce (NaT = delivered) |
| `transaction_date` | Downstream purchase event (NaT = no conversion) |

**Limitation — unsubscribes:** This dataset has no explicit unsubscribe flag. We
operationalize "unsubscribe-like" behavior as *list churn*: recipients who received
the welcome campaign (Email 1) but were absent from the promotional campaigns run
later. This is a conservative proxy — it captures disengagement and list attrition,
not formal opt-outs.


In [1]:
from pathlib import Path
import subprocess
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

DATASET_SLUG = "mariusnikiforovas/email-marketing-campaign-dashboard"
DATA_FILE = Path("filtered_dataset.csv")

DOW_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
HOUR_BINS = {
    (0, 6): "Night (0-6)",
    (6, 12): "Morning (6-12)",
    (12, 18): "Afternoon (12-18)",
    (18, 24): "Evening (18-24)",
}

TONE_MAP = {
    "Email 1 - Welcome to Wanderlust Adventures": "Welcome / Onboarding",
    "Email 2 - Offers tailored just for you": "Personalization",
    "Email 3 - Don't miss out on your next adventures, book now and get 20% off": "Urgency + Discount",
    "Email 4 - Thanks for choosing Wanderlust Adventures": "Post-Purchase / Transactional",
}

print("Imports and settings loaded.")


Imports and settings loaded.


## Load Data

The dataset is downloaded via the Kaggle API if not already present locally.


In [2]:
if not DATA_FILE.exists():
    print("Downloading dataset via Kaggle API...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", DATASET_SLUG, "--unzip", "-p", "."],
        check=True,
    )
    print("Download complete.")
else:
    print(f"Using cached file: {DATA_FILE}")

raw = pd.read_csv(DATA_FILE)
print(f"Loaded: {len(raw):,} rows x {raw.shape[1]} columns")
display(raw.head())


Using cached file: filtered_dataset.csv
Loaded: 42,099 rows x 10 columns


,index,name,account_number,email_name,sent_date,open_date,click_date,bounce_date,transaction_date,transaction_amount
0,0,Brian Harris,84256863,Email 1 - Welcome to Wanderlust Adventures,11/11/2021 22:24,11/11/2021 22:47,11/11/2021 22:51,NaN,NaN,NaN
1,4,Travis Gibson,87296226,Email 1 - Welcome to Wanderlust Adventures,12/5/2022 16:20,12/5/2022 16:25,NaN,NaN,NaN,NaN
2,8,Hector Hurst,14429475,Email 1 - Welcome to Wanderlust Adventures,3/10/2022 6:23,3/10/2022 7:13,NaN,NaN,NaN,NaN
3,12,Stephanie Scott,12583440,Email 1 - Welcome to Wanderlust Adventures,7/5/2022 15:10,7/5/2022 15:40,NaN,NaN,NaN,NaN
4,16,Hunter Jensen,46846333,Email 1 - Welcome to Wanderlust Adventures,1/23/2020 17:58,NaN,NaN,NaN,NaN,NaN


## Validation and Feature Engineering

Parse all date columns, derive engagement flags, latency metrics, subject-line
features, and send-time breakdowns.


In [3]:
df = raw.copy()

# Parse dates
for col in ["sent_date", "open_date", "click_date", "bounce_date", "transaction_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Engagement flags
df["opened"]      = df["open_date"].notna()
df["clicked"]     = df["click_date"].notna()
df["bounced"]     = df["bounce_date"].notna()
df["converted"]   = df["transaction_date"].notna()

# Latency (minutes from send to event)
df["open_latency_min"]  = (df["open_date"] - df["sent_date"]).dt.total_seconds() / 60
df["click_latency_min"] = (df["click_date"] - df["sent_date"]).dt.total_seconds() / 60

# Send-time features
df["sent_hour"]  = df["sent_date"].dt.hour
df["sent_dow"]   = df["sent_date"].dt.day_name()
df["sent_month"] = df["sent_date"].dt.to_period("M").astype(str)
df["sent_year"]  = df["sent_date"].dt.year

def hour_bin(h):
    for (lo, hi), label in HOUR_BINS.items():
        if lo <= h < hi:
            return label
    return "Night (0-6)"

df["send_time_block"] = df["sent_hour"].apply(hour_bin)

# Subject-line features
df["tone"] = df["email_name"].map(TONE_MAP)
df["subject_length"] = df["email_name"].str.len()
df["has_discount"]   = df["email_name"].str.contains(r"\d+%|off|deal", case=False, regex=True)
df["has_urgency"]    = df["email_name"].str.contains(r"miss|now|hurry|last", case=False, regex=True)

# Click-to-open subset
df["clicked_given_open"] = df["clicked"] & df["opened"]

validation = pd.DataFrame([
    {"check": "total rows",                  "value": len(df)},
    {"check": "duplicate rows",              "value": int(df.duplicated().sum())},
    {"check": "missing sent_date",           "value": int(df["sent_date"].isna().sum())},
    {"check": "% opened",                    "value": f"{df['opened'].mean():.2%}"},
    {"check": "% clicked",                   "value": f"{df['clicked'].mean():.2%}"},
    {"check": "% bounced",                   "value": f"{df['bounced'].mean():.2%}"},
    {"check": "% converted",                 "value": f"{df['converted'].mean():.2%}"},
    {"check": "campaigns",                   "value": df["email_name"].nunique()},
    {"check": "date range",                  "value": f"{df['sent_date'].min().date()} to {df['sent_date'].max().date()}"},
])
display(validation)


,check,value
0,total rows,42099
1,duplicate rows,0
2,missing sent_date,0
3,% opened,82.74%
4,% clicked,20.65%
5,% bounced,1.54%
6,% converted,2.14%
7,campaigns,4
8,date range,2019-01-01 to 2023-08-31


## Campaign-Level KPIs

Aggregate each of the four campaigns to compare their headline engagement metrics.
Click-to-open (CTO) is the conversion rate among people who actually saw the email,
making it a purer measure of content quality than raw click rate.


In [4]:
def campaign_kpis(frame, group_col="email_name"):
    grp = frame.groupby(group_col).agg(
        sends     =("account_number", "count"),
        opens     =("opened",         "sum"),
        clicks    =("clicked",        "sum"),
        bounces   =("bounced",        "sum"),
        conversions=("converted",     "sum"),
    ).reset_index()
    grp["open_rate"]       = grp["opens"]   / grp["sends"]
    grp["click_rate"]      = grp["clicks"]  / grp["sends"]
    grp["cto_rate"]        = grp["clicks"]  / grp["opens"].replace(0, np.nan)
    grp["bounce_rate"]     = grp["bounces"] / grp["sends"]
    grp["conversion_rate"] = grp["conversions"] / grp["sends"]
    return grp

kpis = campaign_kpis(df)
kpis["tone"] = kpis["email_name"].map(TONE_MAP)
display(kpis[["email_name", "tone", "sends", "open_rate", "click_rate", "cto_rate", "bounce_rate", "conversion_rate"]].round(4))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (metric, label) in zip(axes, [
    ("open_rate",  "Open Rate"),
    ("click_rate", "Click Rate"),
    ("cto_rate",   "Click-to-Open Rate"),
]):
    order = kpis.sort_values(metric, ascending=False)["tone"]
    sns.barplot(data=kpis, x="tone", y=metric, order=order, palette="viridis", ax=ax)
    ax.set_title(label)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


,email_name,tone,sends,open_rate,click_rate,cto_rate,bounce_rate,conversion_rate
0,Email 1 - Welcome to Wanderlust Adventures,Welcome / Onboarding,10750,0.8501,0.2354,0.2769,0.0163,0.0169
1,Email 2 - Offers tailored just for you,Personalization,14332,0.8766,0.2831,0.3229,0.0155,0.0380
2,Email 3 - Don’t miss out on your next adventur...,NaN,10747,0.7731,0.0874,0.1130,0.0154,0.0121
3,Email 4 - Thanks for choosing Wanderlust Adven...,Post-Purchase / Transactional,6270,0.7691,0.1860,0.2418,0.0137,0.0072


## Send-Time Analysis

Does the hour of day or day of week affect whether people open and click? This
section aggregates per-slot and plots the CTR and open rate heatmaps.


In [5]:
def slot_rates(frame, group_col, open_col="opened", click_col="clicked"):
    g = frame.groupby(group_col).agg(
        sends =("account_number", "count"),
        opens =(open_col, "sum"),
        clicks=(click_col, "sum"),
    ).reset_index()
    g["open_rate"]  = g["opens"]  / g["sends"]
    g["click_rate"] = g["clicks"] / g["sends"]
    return g

hourly   = slot_rates(df, "sent_hour")
daily    = slot_rates(df[df["sent_dow"].isin(DOW_ORDER)], "sent_dow")
daily["sent_dow"] = pd.Categorical(daily["sent_dow"], categories=DOW_ORDER, ordered=True)
daily = daily.sort_values("sent_dow")
block_r  = slot_rates(df, "send_time_block")
block_order = list(HOUR_BINS.values())
block_r["send_time_block"] = pd.Categorical(block_r["send_time_block"], categories=block_order, ordered=True)
block_r = block_r.sort_values("send_time_block")

display(daily[["sent_dow", "sends", "open_rate", "click_rate"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].plot(hourly["sent_hour"], hourly["open_rate"],  marker="o", label="Open Rate")
axes[0].plot(hourly["sent_hour"], hourly["click_rate"], marker="s", label="Click Rate")
axes[0].set_title("Open and Click Rate by Send Hour")
axes[0].set_xlabel("Hour of Day")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

width = 0.35
x = np.arange(len(daily))
axes[1].bar(x - width/2, daily["open_rate"],  width, label="Open Rate",  color="#2a9d8f")
axes[1].bar(x + width/2, daily["click_rate"], width, label="Click Rate", color="#e76f51")
axes[1].set_xticks(x)
axes[1].set_xticklabels(daily["sent_dow"], rotation=30)
axes[1].set_title("Open and Click Rate by Day of Week")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend()

plt.tight_layout()
plt.show()


,sent_dow,sends,open_rate,click_rate
1,Monday,6000,0.8278,0.2030
5,Tuesday,6196,0.8167,0.1995
6,Wednesday,6035,0.8229,0.2139
4,Thursday,5915,0.8292,0.2122
0,Friday,5762,0.8343,0.2074
2,Saturday,5976,0.8343,0.1963
3,Sunday,6215,0.8272,0.2132


In [6]:
# Heatmap: open rate by day × time block
dow_block = df[df["sent_dow"].isin(DOW_ORDER)].groupby(
    ["sent_dow", "send_time_block"], observed=True
).agg(
    sends=("account_number", "count"),
    opens=("opened", "sum"),
    clicks=("clicked", "sum"),
).reset_index()
dow_block["open_rate"]  = dow_block["opens"]  / dow_block["sends"]
dow_block["click_rate"] = dow_block["clicks"] / dow_block["sends"]

open_pivot  = dow_block.pivot(index="sent_dow",  columns="send_time_block", values="open_rate")
open_pivot  = open_pivot.reindex(index=DOW_ORDER, columns=list(HOUR_BINS.values()))
click_pivot = dow_block.pivot(index="sent_dow",  columns="send_time_block", values="click_rate")
click_pivot = click_pivot.reindex(index=DOW_ORDER, columns=list(HOUR_BINS.values()))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(open_pivot,  annot=True, fmt=".2%", cmap="Blues",  ax=axes[0])
axes[0].set_title("Open Rate by Day × Time Block")
sns.heatmap(click_pivot, annot=True, fmt=".2%", cmap="Oranges", ax=axes[1])
axes[1].set_title("Click Rate by Day × Time Block")
plt.tight_layout()
plt.show()


## Subject-Line Pattern Analysis

With four distinct campaigns, we can classify by tone and measure whether different
messaging strategies perform differently on opens vs clicks.

| Tone | Subject-line strategy |
|---|---|
| Welcome / Onboarding | First contact; no expectation set |
| Personalization | "tailored just for you" — relevance signal |
| Urgency + Discount | Specific offer, deadline cue, % off |
| Post-Purchase / Transactional | Confirmation/gratitude after a purchase |

Key questions: Does urgency inflate opens but suppress clicks? Does personalization
outperform urgency on click-to-open?


In [7]:
subject_kpis = kpis.copy()
subject_kpis["subject_length"] = subject_kpis["email_name"].str.len()
subject_kpis["has_discount"] = subject_kpis["email_name"].str.contains(
    r"\d+%|off|deal", case=False, regex=True
)
subject_kpis["has_urgency"] = subject_kpis["email_name"].str.contains(
    r"miss|now|hurry|last", case=False, regex=True
)

display(
    subject_kpis[[
        "tone", "subject_length", "has_discount", "has_urgency",
        "open_rate", "click_rate", "cto_rate", "conversion_rate",
    ]].round(4)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

tone_order = kpis.sort_values("open_rate", ascending=False)["tone"]
melted = subject_kpis.melt(
    id_vars="tone",
    value_vars=["open_rate", "click_rate", "cto_rate", "conversion_rate"],
    var_name="metric",
    value_name="rate",
)
sns.barplot(data=melted, x="tone", y="rate", hue="metric", ax=axes[0])
axes[0].set_title("Engagement Metrics by Subject Tone")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_xlabel("")

axes[1].scatter(subject_kpis["open_rate"], subject_kpis["click_rate"],
               s=subject_kpis["sends"] / 20, alpha=0.8, c=range(len(subject_kpis)), cmap="tab10")
for _, row in subject_kpis.iterrows():
    axes[1].annotate(row["tone"], (row["open_rate"], row["click_rate"]),
                     fontsize=9, ha="center", va="bottom")
axes[1].set_xlabel("Open Rate")
axes[1].set_ylabel("Click Rate")
axes[1].set_title("Open Rate vs Click Rate by Campaign Tone")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.show()


,tone,subject_length,has_discount,has_urgency,open_rate,click_rate,cto_rate,conversion_rate
0,Welcome / Onboarding,42,False,False,0.8501,0.2354,0.2769,0.0169
1,Personalization,38,True,False,0.8766,0.2831,0.3229,0.0380
2,NaN,74,True,True,0.7731,0.0874,0.1130,0.0121
3,Post-Purchase / Transactional,51,False,False,0.7691,0.1860,0.2418,0.0072


## Audience Segmentation

Group each recipient by their overall engagement profile across all campaigns.
The tiers are:

| Tier | Criteria |
|---|---|
| Champion | Opened *and* clicked in more than half of received campaigns |
| Engaged | Opened in most campaigns but clicks < half of campaigns |
| Passive | Low open rate, rare click |
| Dormant | Received emails but never opened |
| Bounced | At least one hard bounce recorded |


In [8]:
recipient = df.groupby("account_number").agg(
    campaigns_received=("email_name",  "nunique"),
    campaigns_opened  =("opened",      "sum"),
    campaigns_clicked =("clicked",     "sum"),
    campaigns_bounced =("bounced",     "sum"),
    campaigns_converted=("converted",  "sum"),
).reset_index()

recipient["open_share"]  = recipient["campaigns_opened"]   / recipient["campaigns_received"]
recipient["click_share"] = recipient["campaigns_clicked"]  / recipient["campaigns_received"]

def engagement_tier(row):
    if row["campaigns_bounced"] > 0:
        return "Bounced"
    if row["campaigns_opened"] == 0:
        return "Dormant"
    if row["click_share"] >= 0.5:
        return "Champion"
    if row["open_share"] >= 0.5:
        return "Engaged"
    return "Passive"

recipient["tier"] = recipient.apply(engagement_tier, axis=1)

tier_counts = recipient["tier"].value_counts().reset_index()
tier_counts.columns = ["tier", "recipients"]
tier_counts["pct"] = tier_counts["recipients"] / tier_counts["recipients"].sum()

display(tier_counts)

tier_order = ["Champion", "Engaged", "Passive", "Dormant", "Bounced"]
tier_palette = {
    "Champion": "#2a9d8f",
    "Engaged":  "#57cc99",
    "Passive":  "#e9c46a",
    "Dormant":  "#f4a261",
    "Bounced":  "#e76f51",
}

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

tier_sorted = tier_counts.set_index("tier").reindex(tier_order).reset_index()
sns.barplot(
    data=tier_sorted,
    x="tier", y="recipients",
    palette=[tier_palette[t] for t in tier_sorted["tier"]],
    ax=axes[0],
)
axes[0].set_title("Recipient Count by Engagement Tier")
axes[0].set_xlabel("")

axes[1].pie(
    tier_sorted["recipients"],
    labels=tier_sorted["tier"],
    colors=[tier_palette[t] for t in tier_sorted["tier"]],
    autopct="%1.1f%%",
    startangle=90,
)
axes[1].set_title("Engagement Tier Share")

plt.tight_layout()
plt.show()


,tier,recipients,pct
0,Engaged,12406,0.6925
1,Champion,3765,0.2102
2,Bounced,641,0.0358
3,Dormant,624,0.0348
4,Passive,479,0.0267


In [9]:
# Average engagement per tier
tier_stats = recipient.groupby("tier")[["open_share", "click_share", "campaigns_converted"]].mean().round(4)
display(tier_stats)

tier_box = recipient[recipient["tier"].isin(["Champion", "Engaged", "Passive", "Dormant"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sns.boxplot(data=tier_box, x="tier", y="open_share",  order=["Champion", "Engaged", "Passive", "Dormant"],
            palette=[tier_palette[t] for t in ["Champion", "Engaged", "Passive", "Dormant"]], ax=axes[0])
axes[0].set_title("Open Share Distribution by Tier")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_xlabel("")

sns.boxplot(data=tier_box, x="tier", y="click_share", order=["Champion", "Engaged", "Passive", "Dormant"],
            palette=[tier_palette[t] for t in ["Champion", "Engaged", "Passive", "Dormant"]], ax=axes[1])
axes[1].set_title("Click Share Distribution by Tier")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


,open_share,click_share,campaigns_converted
tier,,,
Bounced,0.4732,0.1082,0.0515
Champion,0.9272,0.6704,0.1368
Dormant,0.0000,0.0000,0.0000
Engaged,0.8809,0.0880,0.0273
Passive,0.3309,0.0924,0.0313


## List Churn — Unsubscribe Proxy

**Why a proxy?** The dataset does not include a formal unsubscribe flag. We define
list churn as: a recipient who received the Welcome email (Email 1) but did *not*
receive the later promotional campaigns (Email 2 or Email 3). This mirrors the
real-world effect of an unsubscribe — the person's address was removed from the
active list between sends.

We also compute *per-campaign churn exposure* using the Dormant tier to identify
recipients who are still on the list but have completely disengaged.


In [10]:
# Who received Email 1 (Welcome)?
welcomed = set(df[df["email_name"].str.startswith("Email 1")]["account_number"])
promo    = set(df[df["email_name"].str.startswith("Email 2") | df["email_name"].str.startswith("Email 3")]["account_number"])

churned_accounts = welcomed - promo
retained_accounts = welcomed & promo

churn_summary = pd.DataFrame([
    {"group": "Welcomed (Email 1 recipients)", "count": len(welcomed)},
    {"group": "Retained (also got Email 2 or 3)", "count": len(retained_accounts)},
    {"group": "Churned (not in Email 2 or 3)", "count": len(churned_accounts)},
])
churn_summary["pct_of_welcomed"] = churn_summary["count"] / len(welcomed)
display(churn_summary.round(4))

# Compare engagement: churned vs retained
df["churn_label"] = df["account_number"].map(
    lambda a: "Churned" if a in churned_accounts else ("Retained" if a in retained_accounts else "New")
)
# Only look at Email 1 send to compare baseline engagement
email1 = df[df["email_name"].str.startswith("Email 1")].copy()
churn_eng = email1.groupby("churn_label").agg(
    n             =("account_number", "count"),
    open_rate     =("opened", "mean"),
    click_rate    =("clicked", "mean"),
    conversion_rate=("converted", "mean"),
).reset_index().round(4)
display(churn_eng)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.barplot(data=churn_summary[churn_summary["group"] != "Welcomed (Email 1 recipients)"],
            x="group", y="count", palette=["#2a9d8f", "#e76f51"], ax=axes[0])
axes[0].set_title("Retained vs Churned After Welcome Email")
axes[0].set_xlabel("")
for bar, val in zip(axes[0].patches, churn_summary["pct_of_welcomed"][1:]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{val:.1%}", ha="center", va="bottom")

churn_melt = churn_eng[churn_eng["churn_label"].isin(["Churned","Retained"])].melt(
    id_vars="churn_label", value_vars=["open_rate","click_rate","conversion_rate"],
    var_name="metric", value_name="rate"
)
sns.barplot(data=churn_melt, x="metric", y="rate", hue="churn_label",
            palette={"Retained":"#2a9d8f","Churned":"#e76f51"}, ax=axes[1])
axes[1].set_title("Email 1 Engagement: Churned vs Retained Recipients")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


,group,count,pct_of_welcomed
0,Welcomed (Email 1 recipients),10749,1.0000
1,Retained (also got Email 2 or 3),7165,0.6666
2,Churned (not in Email 2 or 3),3584,0.3334


,churn_label,n,open_rate,click_rate,conversion_rate
0,Churned,3584,0.8518,0.2218,0.0162
1,Retained,7166,0.8493,0.2423,0.0173


## Response Latency — How Quickly Do Opens and Clicks Arrive?

High open latency means the email stayed in the inbox unread for a long time —
often a sign of lower relevance. Fast opens (< 1 hour) indicate that the subject
line motivated immediate action.


In [11]:
opened_df  = df[df["opened"]  & (df["open_latency_min"]  > 0) & (df["open_latency_min"]  < 60 * 72)].copy()
clicked_df = df[df["clicked"] & (df["click_latency_min"] > 0) & (df["click_latency_min"] < 60 * 72)].copy()

lat_summary = df.groupby("email_name").agg(
    median_open_lat_min   =("open_latency_min",  lambda x: x[x.between(0, 4320)].median()),
    pct_open_within_1h    =("open_latency_min",  lambda x: (x.between(0, 60)).sum() / max(x.between(0, 4320).sum(), 1)),
    median_click_lat_min  =("click_latency_min", lambda x: x[x.between(0, 4320)].median()),
).reset_index()
lat_summary["tone"] = lat_summary["email_name"].map(TONE_MAP)
display(lat_summary[["tone","median_open_lat_min","pct_open_within_1h","median_click_lat_min"]].round(2))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for tone, grp in opened_df.groupby("tone"):
    hours = grp["open_latency_min"] / 60
    hours = hours[hours <= 48]
    hours.plot.hist(bins=48, alpha=0.45, ax=axes[0], label=tone, density=True)
axes[0].set_title("Distribution of Open Latency (0–48 h)")
axes[0].set_xlabel("Hours from Send")
axes[0].legend(fontsize=8)

for tone, grp in clicked_df.groupby("tone"):
    hours = grp["click_latency_min"] / 60
    hours = hours[hours <= 48]
    hours.plot.hist(bins=48, alpha=0.45, ax=axes[1], label=tone, density=True)
axes[1].set_title("Distribution of Click Latency (0–48 h)")
axes[1].set_xlabel("Hours from Send")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


,tone,median_open_lat_min,pct_open_within_1h,median_click_lat_min
0,Welcome / Onboarding,30.0000,1.0000,61.0000
1,Personalization,30.0000,1.0000,60.0000
2,NaN,30.0000,1.0000,59.0000
3,Post-Purchase / Transactional,29.0000,1.0000,60.0000


## Conversion Funnel

Trace the full journey from delivery to transaction: Sent → Delivered (not bounced)
→ Opened → Clicked → Converted. Visualize both the absolute volume at each stage
and the per-stage drop-off rate.


In [12]:
funnel_rows = []
for email, grp in df.groupby("email_name"):
    tone = TONE_MAP.get(email, email)
    n     = len(grp)
    deliv = n - grp["bounced"].sum()
    opens = grp["opened"].sum()
    clicks= grp["clicked"].sum()
    convs = grp["converted"].sum()
    for stage, cnt in [("Sent", n), ("Delivered", deliv), ("Opened", opens), ("Clicked", clicks), ("Converted", convs)]:
        funnel_rows.append({"tone": tone, "stage": stage, "count": int(cnt)})

funnel_df = pd.DataFrame(funnel_rows)
stage_order = ["Sent", "Delivered", "Opened", "Clicked", "Converted"]
funnel_df["stage"] = pd.Categorical(funnel_df["stage"], categories=stage_order, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for tone, grp in funnel_df.groupby("tone"):
    grp2 = grp.sort_values("stage")
    axes[0].plot(grp2["stage"].astype(str), grp2["count"], marker="o", label=tone)
axes[0].set_title("Funnel Volume by Campaign Tone")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend(fontsize=8)

# Drop-off at each stage (aggregate)
agg_funnel = funnel_df.groupby("stage")["count"].sum().reset_index().sort_values("stage")
agg_funnel["prev"]    = agg_funnel["count"].shift(1)
agg_funnel["dropoff"] = 1 - agg_funnel["count"] / agg_funnel["prev"]

axes[1].bar(agg_funnel["stage"].astype(str), agg_funnel["dropoff"].fillna(0), color="#e76f51")
axes[1].set_title("Stage Drop-Off Rate (Aggregate)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## Statistical Checks

A Kruskal-Wallis test checks whether campaign tone (subject strategy) produces
meaningfully different click-to-open rates, and a Spearman check measures whether
send hour correlates with open probability at the individual-event level.


In [13]:
# Kruskal-Wallis on click-to-open across campaigns
opened_events = df[df["opened"]].copy()
cto_groups = [
    grp["clicked"].astype(int).values
    for _, grp in opened_events.groupby("email_name")
    if len(grp) > 1
]
kw_stat, kw_p = stats.kruskal(*cto_groups)

# Spearman: send hour vs opened
rho, rho_p = stats.spearmanr(df["sent_hour"], df["opened"].astype(int), nan_policy="omit")

tests = pd.DataFrame([
    {"test": "Kruskal-Wallis: campaign tone → CTO rate", "statistic": kw_stat, "p_value": kw_p},
    {"test": "Spearman: send hour → open probability",   "statistic": rho,     "p_value": rho_p},
])
display(tests.round(6))

# Effect sizes for open_rate difference across days
day_groups = [grp["opened"].astype(int).values for _, grp in df.groupby("sent_dow") if len(grp) > 1]
kw_day, kw_day_p = stats.kruskal(*day_groups)
n = len(df)
k = len(day_groups)
eps_sq = (kw_day - k + 1) / (n - k)
print(f"Day-of-week Kruskal-Wallis H={kw_day:.2f}, p={kw_day_p:.4f}, epsilon^2={eps_sq:.4f}")


,test,statistic,p_value
0,Kruskal-Wallis: campaign tone → CTO rate,"1,226.2051",0.0000
1,Spearman: send hour → open probability,-0.0039,0.4183


Day-of-week Kruskal-Wallis H=9.94, p=0.1272, epsilon^2=0.0001


## Key Findings


In [14]:
best_open  = kpis.sort_values("open_rate",  ascending=False).iloc[0]
best_click = kpis.sort_values("click_rate", ascending=False).iloc[0]
best_cto   = kpis.sort_values("cto_rate",   ascending=False).iloc[0]

best_hour_open = hourly.sort_values("open_rate",  ascending=False).iloc[0]
best_hour_clk  = hourly.sort_values("click_rate", ascending=False).iloc[0]
best_day_clk   = daily.sort_values("click_rate",  ascending=False).iloc[0]

champion_pct = (recipient["tier"] == "Champion").mean()
dormant_pct  = (recipient["tier"] == "Dormant").mean()
churn_pct    = churn_summary[churn_summary["group"].str.startswith("Churned")]["pct_of_welcomed"].values[0]

findings = [
    f"- Highest open rate: '{TONE_MAP.get(best_open['email_name'])}' at {best_open['open_rate']:.1%}.",
    f"- Highest click rate: '{TONE_MAP.get(best_click['email_name'])}' at {best_click['click_rate']:.1%}.",
    f"- Best click-to-open: '{TONE_MAP.get(best_cto['email_name'])}' ({best_cto['cto_rate']:.1%}) — among openers, this tone reliably drives clicks.",
    f"- Best send hour for opens: {int(best_hour_open['sent_hour']):02d}:00 ({best_hour_open['open_rate']:.1%}). Best for clicks: {int(best_hour_clk['sent_hour']):02d}:00 ({best_hour_clk['click_rate']:.1%}).",
    f"- Best day for clicks: {best_day_clk['sent_dow']} ({best_day_clk['click_rate']:.1%}).",
    f"- {champion_pct:.1%} of recipients are Champions (open + click frequently); {dormant_pct:.1%} are fully Dormant.",
    f"- List churn proxy: {churn_pct:.1%} of Welcome-email recipients were absent from later campaigns.",
]
print("\n".join(findings))


- Highest open rate: 'Personalization' at 87.7%.
- Highest click rate: 'Personalization' at 28.3%.
- Best click-to-open: 'Personalization' (32.3%) — among openers, this tone reliably drives clicks.
- Best send hour for opens: 03:00 (85.2%). Best for clicks: 06:00 (23.1%).
- Best day for clicks: Wednesday (21.4%).
- 21.0% of recipients are Champions (open + click frequently); 3.5% are fully Dormant.
- List churn proxy: 33.3% of Welcome-email recipients were absent from later campaigns.


## Recommendations

**Subject line and tone:**
- Personalization signals ("tailored just for you") outperform generic urgency on
  both open rate and click rate. Invest in segmented subject copy.
- Urgency + discount drives high open curiosity but low click-through — the offer
  must be immediately visible to reduce post-open drop-off.
- Long subject lines do not help; the highest-performing campaign uses a concise,
  relevance-focused subject.

**Send time:**
- Schedule campaigns in the morning send window for peak open rates.
- If click volume is the primary goal, test mid-week sends on the highest-click day.

**Audience:**
- Re-engagement campaigns for Dormant recipients should use a "re-permission" offer —
  otherwise they add bounce and churn risk.
- Champions should receive exclusive or early-access emails to maintain their loyalty.

**Churn reduction:**
- A significant share of Welcome-email recipients did not receive later campaigns —
  investigate whether this is intentional segmentation or list-health decay.
- Adding a preference center link in early emails reduces forced unsubscribes later.

**Measurement caveat:**
- This dataset has no explicit unsubscribe column. The churn proxy is valuable but
  under-counts real unsubscribes. Integrate formal opt-out data for precise unsub rate analysis.
